# 🏁 H2GP Real-Time Telemetry Dashboard

**Mac Alternative to Excel Data Streamer**

This Jupyter notebook provides real-time visualization of your H2GP fuel cell racing car telemetry data, streaming directly from your Arduino receiver.

## 📊 Features
- **Real-time data streaming** from WebSocket server
- **Interactive plots** with zoom, pan, and hover details
- **Live telemetry monitoring** of voltage, current, and timing
- **CSV data analysis** for historical race data
- **Export capabilities** for race reports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import websocket
import json
import threading
import time
from datetime import datetime
from IPython.display import display, clear_output
import ipywidgets as widgets
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Set up styling
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
print("✅ Libraries loaded successfully!")
print("🔋 H2GP Telemetry Dashboard Ready")

## ⚙️ Configuration

Configure the connection to your H2GP data streaming server:

In [ ]:
# H2GP Data Streaming Configuration
WEBSOCKET_URL = "ws://localhost:8084"  # Your H2GP data streaming server
CSV_LOG_PATH = "logs/"  # Path to CSV log files
MAX_DATA_POINTS = 1000  # Maximum points to keep in memory for real-time plots

# Initialize data storage for real-time streaming
telemetry_data = {
    'timestamp': deque(maxlen=MAX_DATA_POINTS),
    'battery_voltage': deque(maxlen=MAX_DATA_POINTS),
    'battery_current': deque(maxlen=MAX_DATA_POINTS),
    'fuel_cell_voltage': deque(maxlen=MAX_DATA_POINTS),
    'battery_voltage_2nd': deque(maxlen=MAX_DATA_POINTS),
    'purge_interval': deque(maxlen=MAX_DATA_POINTS)
}

# Connection status
is_connected = False
ws_connection = None

print("⚙️ Configuration set for H2GP telemetry streaming")
print(f"📡 WebSocket URL: {WEBSOCKET_URL}")
print(f"📁 CSV Log Path: {CSV_LOG_PATH}")

## 🔌 Real-Time WebSocket Connection

Connect to your H2GP data streaming server for live telemetry:

In [ ]:
def on_message(ws, message):
    """Handle incoming H2GP telemetry data"""
    try:
        data = json.loads(message)
        
        # Add timestamp
        current_time = datetime.now()
        telemetry_data['timestamp'].append(current_time)
        
        # Store telemetry values
        telemetry_data['battery_voltage'].append(data.get('battery_voltage', 0))
        telemetry_data['battery_current'].append(data.get('battery_current', 0))
        telemetry_data['fuel_cell_voltage'].append(data.get('fuel_cell_voltage', 0))
        telemetry_data['battery_voltage_2nd'].append(data.get('battery_voltage_2nd', 0))
        telemetry_data['purge_interval'].append(data.get('purge_interval', 0))
        
    except json.JSONDecodeError:
        print(f"⚠️ Invalid JSON received: {message}")
    except Exception as e:
        print(f"❌ Error processing data: {e}")

def on_error(ws, error):
    print(f"❌ WebSocket error: {error}")

def on_close(ws, close_status_code, close_msg):
    global is_connected
    is_connected = False
    print("🔌 Disconnected from H2GP data stream")

def on_open(ws):
    global is_connected
    is_connected = True
    print("✅ Connected to H2GP telemetry stream!")
    print("🏁 Ready to receive live racing data")

def connect_to_h2gp_stream():
    """Connect to H2GP data streaming server"""
    global ws_connection
    try:
        ws_connection = websocket.WebSocketApp(
            WEBSOCKET_URL,
            on_message=on_message,
            on_error=on_error,
            on_close=on_close,
            on_open=on_open
        )
        
        # Start connection in background thread
        def run_connection():
            ws_connection.run_forever()
        
        connection_thread = threading.Thread(target=run_connection, daemon=True)
        connection_thread.start()
        
        print("🚀 Connecting to H2GP telemetry server...")
        time.sleep(2)  # Give it time to connect
        
        if is_connected:
            print("🎉 Successfully connected to live H2GP data!")
        else:
            print("⚠️ Connection attempt in progress...")
            
    except Exception as e:
        print(f"❌ Failed to connect: {e}")
        print("💡 Make sure your H2GP server is running: node server.js")

# Connect to H2GP data stream
connect_to_h2gp_stream()

## 📈 Real-Time Telemetry Visualization

Live interactive plots of your H2GP racing car telemetry:

In [ ]:
def create_realtime_dashboard():
    """Create real-time H2GP telemetry dashboard"""
    
    # Create subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=(
            'Battery Voltage (V)', 'Fuel Cell Voltage (V)',
            'Battery Current (A)', 'Battery Voltage 2nd (V)', 
            'Purge Interval (s)', 'System Overview'
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"colspan": 1}]],
        vertical_spacing=0.08
    )
    
    if len(telemetry_data['timestamp']) > 0:
        timestamps = list(telemetry_data['timestamp'])
        
        # Battery Voltage
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=list(telemetry_data['battery_voltage']),
                mode='lines+markers',
                name='Battery V',
                line=dict(color='#FF6B6B', width=2),
                marker=dict(size=4)
            ),
            row=1, col=1
        )
        
        # Fuel Cell Voltage
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=list(telemetry_data['fuel_cell_voltage']),
                mode='lines+markers',
                name='Fuel Cell V',
                line=dict(color='#4ECDC4', width=2),
                marker=dict(size=4)
            ),
            row=1, col=2
        )
        
        # Battery Current
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=list(telemetry_data['battery_current']),
                mode='lines+markers',
                name='Battery I',
                line=dict(color='#45B7D1', width=2),
                marker=dict(size=4)
            ),
            row=2, col=1
        )
        
        # Battery Voltage 2nd
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=list(telemetry_data['battery_voltage_2nd']),
                mode='lines+markers',
                name='Battery V2',
                line=dict(color='#F7DC6F', width=2),
                marker=dict(size=4)
            ),
            row=2, col=2
        )
        
        # Purge Interval
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=list(telemetry_data['purge_interval']),
                mode='lines+markers',
                name='Purge Interval',
                line=dict(color='#BB8FCE', width=2),
                marker=dict(size=4)
            ),
            row=3, col=1
        )
        
        # System Overview - Power (V * I)
        power_data = [v * i for v, i in zip(telemetry_data['battery_voltage'], telemetry_data['battery_current'])]
        fig.add_trace(
            go.Scatter(
                x=timestamps, 
                y=power_data,
                mode='lines+markers',
                name='Power (W)',
                line=dict(color='#E74C3C', width=3),
                marker=dict(size=6)
            ),
            row=3, col=2
        )
        
        # Latest values display
        latest_idx = -1
        latest_values = {
            'Battery Voltage': f"{telemetry_data['battery_voltage'][latest_idx]:.3f} V",
            'Fuel Cell Voltage': f"{telemetry_data['fuel_cell_voltage'][latest_idx]:.3f} V",
            'Battery Current': f"{telemetry_data['battery_current'][latest_idx]:.3f} A",
            'Power': f"{power_data[latest_idx]:.3f} W",
            'Data Points': f"{len(timestamps)}"
        }
        
        print("\n🔋 Latest H2GP Telemetry:")
        for key, value in latest_values.items():
            print(f"   {key}: {value}")
    
    else:
        # No data available
        fig.add_annotation(
            text="⏳ Waiting for H2GP telemetry data...<br>Make sure your server is running: node server.js",
            x=0.5, y=0.5,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=16, color="gray")
        )
    
    # Update layout
    fig.update_layout(
        title={
            'text': '🏁 H2GP Real-Time Telemetry Dashboard',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20}
        },
        height=800,
        showlegend=True,
        template='plotly_white'
    )
    
    # Update axes labels
    fig.update_yaxes(title_text="Voltage (V)", row=1, col=1)
    fig.update_yaxes(title_text="Voltage (V)", row=1, col=2)
    fig.update_yaxes(title_text="Current (A)", row=2, col=1)
    fig.update_yaxes(title_text="Voltage (V)", row=2, col=2)
    fig.update_yaxes(title_text="Time (s)", row=3, col=1)
    fig.update_yaxes(title_text="Power (W)", row=3, col=2)
    
    return fig

# Create and display the dashboard
print("🎯 Creating H2GP Real-Time Dashboard...")
dashboard_fig = create_realtime_dashboard()
dashboard_fig.show()

print("\n🔄 Dashboard created! It will auto-update as new data arrives.")
print("💡 Run the next cell to start live updates")

## 🔄 Live Dashboard Updates

Run this cell to start live updating of the dashboard:

In [ ]:
def start_live_updates(update_interval=2):
    """Start live dashboard updates"""
    
    update_button = widgets.Button(
        description='🔄 Update Dashboard',
        button_style='success',
        tooltip='Click to refresh telemetry data'
    )
    
    stop_button = widgets.Button(
        description='⏹️ Stop Updates',
        button_style='danger',
        tooltip='Stop live updates'
    )
    
    status_label = widgets.Label(value="📡 Live H2GP telemetry streaming...")
    
    # Control flags
    updating = {'active': True}
    
    def update_dashboard(b):
        """Manual dashboard update"""
        clear_output(wait=True)
        
        # Show controls
        display(widgets.HBox([update_button, stop_button]))
        display(status_label)
        
        # Show updated dashboard
        new_fig = create_realtime_dashboard()
        new_fig.show()
        
        # Connection status
        if is_connected:
            status_label.value = f"✅ Connected - {len(telemetry_data['timestamp'])} data points received"
        else:
            status_label.value = "❌ Disconnected - Check H2GP server"
    
    def stop_updates(b):
        """Stop live updates"""
        updating['active'] = False
        status_label.value = "⏹️ Live updates stopped"
        if ws_connection:
            ws_connection.close()
    
    def auto_update():
        """Automatic dashboard updates"""
        while updating['active']:
            time.sleep(update_interval)
            if updating['active'] and len(telemetry_data['timestamp']) > 0:
                update_dashboard(None)
    
    # Set button callbacks
    update_button.on_click(update_dashboard)
    stop_button.on_click(stop_updates)
    
    # Show controls
    display(widgets.HBox([update_button, stop_button]))
    display(status_label)
    
    # Start auto-updates in background
    auto_thread = threading.Thread(target=auto_update, daemon=True)
    auto_thread.start()
    
    # Initial dashboard display
    update_dashboard(None)

# Start live dashboard
print("🚀 Starting live H2GP telemetry dashboard...")
start_live_updates(update_interval=3)  # Update every 3 seconds

## 📊 Historical Data Analysis

Load and analyze your H2GP CSV log files:

In [ ]:
import glob
import os

def load_h2gp_csv_data():
    """Load H2GP CSV telemetry files"""
    
    # Find all CSV files
    csv_files = glob.glob(f"{CSV_LOG_PATH}H2GP_Telemetry_*.csv")
    
    if not csv_files:
        print("⚠️ No H2GP CSV files found in logs/ directory")
        return None
    
    # Load the most recent CSV file
    latest_file = max(csv_files, key=os.path.getctime)
    print(f"📁 Loading: {latest_file}")
    
    try:
        df = pd.read_csv(latest_file)
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        
        print(f"✅ Loaded {len(df)} telemetry records")
        print(f"📅 Date Range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return None

def analyze_h2gp_performance(df):
    """Analyze H2GP performance metrics"""
    
    if df is None or len(df) == 0:
        print("⚠️ No data to analyze")
        return
    
    # Calculate performance metrics
    df['Power_W'] = df['Battery_Voltage_V'] * df['Battery_Current_A']
    df['Energy_Wh'] = df['Power_W'] * (1/3600)  # Approximate energy per second
    
    # Performance summary
    print("\n🏁 H2GP Performance Analysis:")
    print(f"   📊 Total Data Points: {len(df)}")
    print(f"   ⚡ Max Power: {df['Power_W'].max():.3f} W")
    print(f"   🔋 Avg Battery Voltage: {df['Battery_Voltage_V'].mean():.3f} V")
    print(f"   ⚡ Avg Current Draw: {df['Battery_Current_A'].mean():.3f} A")
    print(f"   🔋 Fuel Cell Voltage Range: {df['Fuel_Cell_Voltage_V'].min():.3f} - {df['Fuel_Cell_Voltage_V'].max():.3f} V")
    print(f"   💨 Avg Purge Interval: {df['Purge_Interval_s'].mean():.3f} s")
    
    # Create comprehensive analysis plots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Voltage Trends', 'Current & Power', 'Performance Distribution', 'System Timeline'),
        specs=[[{"secondary_y": True}, {"secondary_y": True}],
               [{"type": "histogram"}, {"secondary_y": False}]]
    )
    
    # Voltage trends
    fig.add_trace(
        go.Scatter(x=df['Timestamp'], y=df['Battery_Voltage_V'], name='Battery V', line=dict(color='red')),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df['Timestamp'], y=df['Fuel_Cell_Voltage_V'], name='Fuel Cell V', line=dict(color='blue')),
        row=1, col=1, secondary_y=True
    )
    
    # Current & Power
    fig.add_trace(
        go.Scatter(x=df['Timestamp'], y=df['Battery_Current_A'], name='Current A', line=dict(color='green')),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(x=df['Timestamp'], y=df['Power_W'], name='Power W', line=dict(color='orange')),
        row=1, col=2, secondary_y=True
    )
    
    # Performance distribution
    fig.add_trace(
        go.Histogram(x=df['Power_W'], name='Power Distribution', nbinsx=30),
        row=2, col=1
    )
    
    # System timeline
    fig.add_trace(
        go.Scatter(x=df['Timestamp'], y=df['Purge_Interval_s'], name='Purge Interval', 
                  mode='lines+markers', line=dict(color='purple')),
        row=2, col=2
    )
    
    fig.update_layout(
        title='📊 H2GP Historical Performance Analysis',
        height=600,
        showlegend=True
    )
    
    fig.show()
    
    return df

# Load and analyze H2GP data
h2gp_data = load_h2gp_csv_data()
if h2gp_data is not None:
    analyze_h2gp_performance(h2gp_data)
else:
    print("💡 Start your H2GP data streaming server to generate CSV files:")
    print("   cd serial-data-streamer && node server.js")

## 📤 Export and Reporting

Export your H2GP telemetry data for race reports:

In [ ]:
def export_h2gp_report(df, filename_prefix="H2GP_Race_Report"):
    """Export H2GP race report"""
    
    if df is None:
        print("⚠️ No data to export")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export processed CSV
    csv_filename = f"{filename_prefix}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"✅ Exported CSV: {csv_filename}")
    
    # Create summary report
    summary = {
        'Race_Date': timestamp,
        'Total_Records': len(df),
        'Max_Power_W': df['Power_W'].max(),
        'Avg_Battery_V': df['Battery_Voltage_V'].mean(),
        'Avg_Current_A': df['Battery_Current_A'].mean(),
        'Max_Fuel_Cell_V': df['Fuel_Cell_Voltage_V'].max(),
        'Min_Fuel_Cell_V': df['Fuel_Cell_Voltage_V'].min()
    }
    
    # Export summary JSON
    json_filename = f"{filename_prefix}_Summary_{timestamp}.json"
    with open(json_filename, 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    
    print(f"📊 Performance Summary:")
    for key, value in summary.items():
        print(f"   {key}: {value}")
    
    print(f"\n📁 Files exported:")
    print(f"   📄 {csv_filename}")
    print(f"   📊 {json_filename}")
    
    return csv_filename, json_filename

# Export current telemetry data
if 'h2gp_data' in locals() and h2gp_data is not None:
    export_h2gp_report(h2gp_data)
else:
    print("💡 Load H2GP data first before exporting")
    print("   Run the Historical Data Analysis cell above")